# matvec — worked example 2: Apply the Same Matrix to Multiple Vectors Without Batching

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matvec`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When you need to apply a fixed weight matrix `W: (M, N)` to many different vectors, the cleanest approach is to stack the vectors into a matrix `X: (K, N)` and use a single matmul `X @ W.T` for the batch. But understanding the per-vector `W @ x` form first clarifies what each row of the batch output represents — one matvec application per sample.

## Worked solution

**Step 1 — build three input vectors and a shared weight matrix.**
We create `W: (2, 3)` (maps 3-D to 2-D) and three separate 1-D vectors of shape `(3,)`.

**Step 2 — apply matvec to each vector in a loop.**
For each vector `x`, `W @ x` produces a `(2,)` vector. We collect the results in a list and stack them.

**Step 3 — verify against the batched form.**
Stacking the vectors into `X: (3, 3)` and computing `X @ W.T` gives a `(3, 2)` matrix. Each row should match the corresponding loop result.

**Step 4 — confirm shapes and numerical agreement.**
We verify that both approaches give identical outputs, cementing the equivalence between repeated matvec and batched matmul.

In [ ]:
import torch as t

t.manual_seed(7)
W = t.randn(2, 3)  # maps 3-D input to 2-D output
vectors = [t.randn(3) for _ in range(4)]

# Loop: one matvec per vector
loop_results = []
for x in vectors:
    y = W @ x   # (2,3)@(3,) -> (2,)
    loop_results.append(y)
loop_out = t.stack(loop_results)  # (4, 2)

# Batched: stack into matrix and matmul
X = t.stack(vectors)              # (4, 3)
batch_out = X @ W.T               # (4,3)@(3,2) -> (4,2)

print(f"Loop output shape:  {loop_out.shape}")
print(f"Batch output shape: {batch_out.shape}")
print(f"Numerically equal:  {t.allclose(loop_out, batch_out, atol=1e-5)}")
print(f"\nFirst row (loop):   {loop_out[0].round(decimals=4)}")
print(f"First row (batch):  {batch_out[0].round(decimals=4)}")